# ML Baseline: Predict next-day extreme loss (SPY)

**Updated:** 2026-03-10  
**Goal:** Build a minimal time-series ML pipeline (no leakage) and diagnose why a baseline may fail on rare events.

We define the label using a train-only quantile threshold on next-day loss:
- `loss_t = -ret_t`
- `y_t = 1{ loss_{t+1} > q(label_q) }`

## 1) Setup (imports)

We use:
- `pandas/numpy` for time-series features
- `scikit-learn` for a baseline classifier (Logistic Regression)

In [52]:
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score

## 2) Data + feature engineering (price → returns → loss)

**Input:** daily adjusted price series (`date`, `price`)  
We compute:
- log return: $r_t = \log(P_t) - \log(P_{t-1}) $
- loss: $ L_t = -r_t $

**Features at time $t$:**
- `ret_t`: today’s log return
- `vol20_t`: rolling 20-day volatility of returns
- `var250_t`: rolling 250-day 1% quantile of returns (left-tail risk)

**Label (target):**  
$ y_t = 1\{ L_{t+1} > q_{\text{label\_q}}(L_{t+1}) \} $


Important: the quantile threshold is computed on train only to avoid leakage.

In [53]:
label_q = 0.90  # we tried 0.95 and 0.90 to avoid "no positives" in test

df = pd.read_csv("../data/price_SPY.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

price = df["price"].astype(float)

df_feat = pd.DataFrame(index=df.index)
df_feat["ret"] = np.log(price).diff()
df_feat["loss"] = -df_feat["ret"]

df_feat["vol20"] = df_feat["ret"].rolling(20).std()
df_feat["var250"] = df_feat["ret"].rolling(250).quantile(0.01)

df_feat["loss_t1"] = df_feat["loss"].shift(-1)

feat_cols = ["ret", "vol20", "var250"]
df_feat = df_feat.dropna(subset=feat_cols + ["loss_t1"]).copy()



## 3) Time-based split + leakage-safe label

**Split:** chronological 80/20 split (no shuffle).  
Shuffling time series can leak future information into training.

**Label definition (predicting tomorrow):**
- We predict whether tomorrow is an extreme-loss day.
- Define `loss_t = -ret_t` and `loss_t1 = loss_{t+1}` via shifting.
- Compute the extreme-loss cutoff on the training set only:
  - `thr = quantile(train.loss_t1, label_q)`
- Then define labels:
  - `y_t = 1{ loss_t1 > thr }`

We print class balance (positives in train/test) to understand rarity.

In [54]:
split = int(len(df_feat) * 0.8)
train = df_feat.iloc[:split]
test = df_feat.iloc[split:]

# train-only threshold (avoid leakage)
thr = train["loss_t1"].quantile(label_q)

X_train = train[feat_cols].to_numpy()
X_test = test[feat_cols].to_numpy()

y_train = (train["loss_t1"] > thr).astype(int).to_numpy()
y_test = (test["loss_t1"] > thr).astype(int).to_numpy()

print(f"thr (train-only, q={label_q}) =", float(thr))
print("train positives:", int(y_train.sum()), "/", len(y_train), f"({y_train.mean():.4f})")
print("test positives :", int(y_test.sum()), "/", len(y_test), f"({y_test.mean():.4f})")

thr (train-only, q=0.9) = 0.012771045393253737
train positives: 81 / 804 (0.1007)
test positives : 8 / 201 (0.0398)


## 4) Model + probability diagnostics

**Model:** Logistic Regression baseline with `StandardScaler`
- scaling stabilizes optimization and makes coefficients comparable across features
- this baseline checks whether simple signals exist in our features

**Diagnostics:**
Before choosing a classification threshold, we inspect predicted probabilities:
- distribution summary (min/median/max)
- mean probability for `y=0` vs `y=1`
- ROC-AUC (only meaningful if both classes exist in test)

If probabilities overlap heavily or `mean proba(y=1) ≤ mean proba(y=0)`,
the baseline is not separating the rare event well.

In [55]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, solver="lbfgs")  # (we also tested class_weight="balanced")
)
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]

print("\n=== proba summary ===")
print(pd.Series(proba).describe())

print("\n=== mean proba by class ===")
proba_s = pd.Series(proba, index=test.index)
y_s = pd.Series(y_test, index=test.index)
print("mean proba (y=0):", float(proba_s[y_s==0].mean()))
print("mean proba (y=1):", float(proba_s[y_s==1].mean()))

if len(np.unique(y_test)) == 2:
    print("ROC-AUC:", roc_auc_score(y_test, proba))
else:
    print("ROC-AUC: not defined (only one class in y_test)")


=== proba summary ===
count    201.000000
mean       0.087284
std        0.007812
min        0.073178
25%        0.081719
50%        0.086192
75%        0.092805
max        0.113025
dtype: float64

=== mean proba by class ===
mean proba (y=0): 0.08736911191309613
mean proba (y=1): 0.08522634844706838
ROC-AUC: 0.4514248704663213


## 5) Threshold sweep (precision–recall trade-off)

Rare-event detection depends heavily on the decision threshold.

For each threshold $t$, we compute:
- `TP, FP, FN, TN`
- precision $= \frac{TP}{TP+FP}$
- recall $= \frac{TP}{TP+FN}$

Interpretation:
- Higher recall catches more extreme-loss days but may increase false alarms (FP).
- Precision–recall trade-off is expected in risk alerting systems.

In [56]:
thresholds = [0.5, 0.3, 0.2, 0.1, 0.05]

print("\n=== Threshold sweep on test ===")
print("thr  pred_pos  TP  FP  FN  TN  precision  recall")

for t in thresholds:
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    pred_pos = int(pred.sum())

    print(f"{t:>4.2f} {pred_pos:>8d} {tp:>3d} {fp:>3d} {fn:>3d} {tn:>3d} "
          f"{precision:>9.4f} {recall:>7.4f}")


=== Threshold sweep on test ===
thr  pred_pos  TP  FP  FN  TN  precision  recall
0.50        0   0   0   8 193    0.0000  0.0000
0.30        0   0   0   8 193    0.0000  0.0000
0.20        0   0   0   8 193    0.0000  0.0000
0.10       13   0  13   8 180    0.0000  0.0000
0.05      201   8 193   0   0    0.0398  1.0000


## 6) Summary
- Time-series split (no shuffle) + train-only threshold prevents leakage.
- Rare-event classification is highly sensitive to thresholds and class imbalance.
- With simple features (ret/vol20/var250) + logistic regression, predicted probabilities were narrowly distributed and did not separate y=1 vs y=0 well in this run.

---

# ML Baseline: Predict next-day extreme loss (SPY) using lagged returns

**Updated:** 2026-03-16  
**Goal:** Build a simple time-series classification baseline to predict **next-day extreme loss** using
- lagged returns (`ret_lag1`…`ret_lag5`)
- volatility state (`vol20`)
- tail state (`var250`)

Evaluate with **rare-event friendly** metrics:
- ROC-AUC (ranking quality)
- Alert budget metrics: **Recall@K** (top-K alerts)

**Key idea:**
Features use information available at time $t$ only, while the label is defined using $L_{t+1}$ (next day).

## 1) Load data + Build features/label 

**Feature/label definition (leakage-safe):**

- Log return and loss:
  - $r_t = \log(P_t) - \log(P_{t-1})$
  - $L_t = -r_t$

- Next-day label:
  - $y_t = 1\{L_{t+1} > \mathrm{thr}\}$

- Train-only threshold:
  - $\mathrm{thr}$ is computed **only on train** using the label quantile $\mathrm{label\_q}$.
  - Test labels use the same fixed $\mathrm{thr}$.

In [57]:
import numpy as np
import pandas as pd

# --- knobs ---
label_q = 0.90   # top 10% tomorrow-loss days are positives
max_lag = 5

# 1) Load SPY prices
df = pd.read_csv("../data/price_SPY.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

price = df["price"].astype(float)

# 2) Build returns / loss
feat = pd.DataFrame(index=df.index)
feat["ret"] = np.log(price).diff()
feat["loss"] = -feat["ret"]

# 3) State features (using info up to time t)
feat["vol20"] = feat["ret"].rolling(20).std()
feat["var250"] = feat["ret"].rolling(250).quantile(0.01)

# 4) Lagged return features (past-only)
for k in range(1, max_lag + 1):
    feat[f"ret_lag{k}"] = feat["ret"].shift(k)

# 5) Label target = next-day loss
feat["loss_t1"] = feat["loss"].shift(-1)

feat_cols = ["ret", "vol20", "var250"] + [f"ret_lag{k}" for k in range(1, max_lag + 1)]

# Drop NaNs from diff/rolling/shift
feat = feat.dropna(subset=feat_cols + ["loss_t1"]).copy()

# 6) Time-based split (80/20)
split = int(len(feat) * 0.8)
train = feat.iloc[:split]
test = feat.iloc[split:]

# Train-only threshold (avoid leakage)
thr = train["loss_t1"].quantile(label_q)

y_train = (train["loss_t1"] > thr).astype(int).to_numpy()
y_test  = (test["loss_t1"] > thr).astype(int).to_numpy()

X_train = train[feat_cols].to_numpy()
X_test  = test[feat_cols].to_numpy()

print("thr =", float(thr))
print("train positives:", int(y_train.sum()), "/", len(y_train), f"({y_train.mean():.4f})")
print("test positives :", int(y_test.sum()), "/", len(y_test), f"({y_test.mean():.4f})")

thr = 0.012771045393253737
train positives: 81 / 804 (0.1007)
test positives : 8 / 201 (0.0398)


## 2) Train models + Evaluate (ROC-AUC + Recall@K)

### Models
We compare two simple baselines:

- **Logistic Regression (linear baseline)**  
  A scaled linear model that outputs a probability score $\hat p_t$.

- **Random Forest (non-linear baseline)**  
  A tree-ensemble that can capture non-linear patterns and feature interactions
  (e.g., volatility regime + lagged returns).

### Evaluation metrics
Because “extreme loss” events are rare, we avoid relying on accuracy.

- **ROC-AUC (ranking quality)**  
  Measures whether the model tends to assign higher scores to positives than negatives.
  If the test set contains only one class, ROC-AUC is not defined.

- **Alert-budget metric: Recall@K**  
  In practice, we often can only send **K alerts per day** (or per period).  
  We rank days by predicted score $\hat p_t$ and look at the top-K.

  $$ \mathrm{Recall@K} = \frac{\#\{\text{true positives among top-K}\}}{\#\{\text{all true positives}\}} $$

  Interpretation:
  - Recall@10 answers: “If we send 10 alerts, what fraction of extreme-loss days do we catch?”
  - Higher Recall@K is better for risk alerting (but may increase false alarms).

### Output to inspect
- ROC-AUC for Logit and RF
- Recall@K for K ∈ {5, 10, 20, 50}

In [58]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def recall_at_k(y_true, scores, k):
    total_pos = int(np.sum(y_true))
    if total_pos == 0:
        return np.nan
    k = min(k, len(scores))
    idx = np.argsort(scores)[::-1][:k]
    tp = int(np.sum(y_true[idx]))
    return tp / total_pos

# --- Logistic baseline ---
logit = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=4000, solver="lbfgs", class_weight="balanced"),
)
logit.fit(X_train, y_train)
proba_logit = logit.predict_proba(X_test)[:, 1]

print("[Logit] ROC-AUC:", roc_auc_score(y_test, proba_logit) if len(np.unique(y_test)) == 2 else "NA")

print("[Logit] Recall@K:")
for k in [5, 10, 20, 50]:
    print(f"  Recall@{k}: {recall_at_k(y_test, proba_logit, k):.4f}")

# --- RandomForest baseline ---
rf = RandomForestClassifier(
    n_estimators=500,
    random_state=0,
    class_weight="balanced_subsample",
    min_samples_leaf=5,
)
rf.fit(X_train, y_train)
proba_rf = rf.predict_proba(X_test)[:, 1]

print("\n[RF] ROC-AUC:", roc_auc_score(y_test, proba_rf) if len(np.unique(y_test)) == 2 else "NA")

print("[RF] Recall@K:")
for k in [5, 10, 20, 50]:
    print(f"  Recall@{k}: {recall_at_k(y_test, proba_rf, k):.4f}")

[Logit] ROC-AUC: 0.4533678756476684
[Logit] Recall@K:
  Recall@5: 0.0000
  Recall@10: 0.0000
  Recall@20: 0.0000
  Recall@50: 0.2500

[RF] ROC-AUC: 0.694300518134715
[RF] Recall@K:
  Recall@5: 0.0000
  Recall@10: 0.1250
  Recall@20: 0.1250
  Recall@50: 0.5000


## 3) Summary

- If RF improves ROC-AUC and Recall@K, it suggests non-linear interactions among lagged returns and volatility/state features.
- Because positives are rare, top-K alert metrics are more practical than a single fixed threshold.

---

## ML Baseline: Alert-budget metrics + Expanding walk-forward

**Updated:** 2026-03-17  
**Goal:**
- Add **alert-budget evaluation** for rare events using Precision@K / Recall@K.
- Move beyond a single 80/20 split by using an **expanding walk-forward** evaluation.
- Keep everything **leakage-safe**:
  - Labels use next-day loss, and the threshold is computed on train only (per split / per fold).

### What we will produce
- Precision@K / Recall@K results for Logistic Regression and RandomForest  
-  A fold-by-fold walk-forward table: (threshold per fold, AUC, Precision@K, Recall@K)

## 1) Import

In [59]:
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

## 2) Alert-budget metrics

### Alert-budget idea (Top-K alerts)
For rare events, a fixed probability threshold (e.g., 0.5) can be impractical.  
Instead, we assume we can send only **K alerts** per day/week (or per evaluation block).

- **Recall@K:** among all true positives, how many are captured within Top-K scores  
- **Precision@K:** among Top-K alerts, what fraction are true positives

In [60]:
def build_features(df: pd.DataFrame, max_lag: int = 5) -> pd.DataFrame:
    """
    Build leakage-safe features at time t (using info up to t).
    Label is based on loss at t+1 (next day): loss_t1 = loss.shift(-1).
    """
    df = df.copy()
    price = df["price"].astype(float)

    feat = pd.DataFrame(index=df.index)
    feat["ret"] = np.log(price).diff()
    feat["loss"] = -feat["ret"]

    # state features
    feat["vol20"] = feat["ret"].rolling(20).std()
    feat["var250"] = feat["ret"].rolling(250).quantile(0.01)

    # lagged returns (past info only)
    for k in range(1, max_lag + 1):
        feat[f"ret_lag{k}"] = feat["ret"].shift(k)

    # label uses tomorrow's loss
    feat["loss_t1"] = feat["loss"].shift(-1)
    return feat


def recall_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
    """
    recall@k = TP_in_topk / total_positives
    """
    total_pos = int(y_true.sum())
    if total_pos == 0:
        return float("nan")
    k = min(k, len(scores))
    idx = np.argsort(scores)[::-1][:k]
    tp = int(y_true[idx].sum())
    return tp / total_pos


def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
    """
    precision@k = TP_in_topk / k
    """
    if len(scores) == 0:
        return float("nan")
    k = min(k, len(scores))
    idx = np.argsort(scores)[::-1][:k]
    tp = int(y_true[idx].sum())
    return tp / k

## 3) Single split (80/20) baseline (Logit vs RF)

We first reproduce a simple chronological 80/20 split (no shuffle).  
Important: the label threshold **thr** must be computed on **train only** to avoid leakage.

We will compare:
- Logistic Regression (scaled) vs RandomForest
- ROC-AUC + Precision@K / Recall@K for K ∈ {5, 10, 20, 50}

In [61]:

# --- knobs ---
label_q = 0.90   # top 10% tomorrow-loss days as positives (train-only threshold)
max_lag = 5
Ks = [5, 10, 20, 50]

# Load SPY
df = pd.read_csv("../data/price_SPY.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

feat = build_features(df, max_lag=max_lag)
feat_cols = ["ret", "vol20", "var250"] + [f"ret_lag{k}" for k in range(1, max_lag + 1)]
feat = feat.dropna(subset=feat_cols + ["loss_t1"]).copy()

# chronological 80/20 split
split = int(len(feat) * 0.8)
train = feat.iloc[:split]
test = feat.iloc[split:]

# train-only threshold to avoid leakage
thr = train["loss_t1"].quantile(label_q)

y_train = (train["loss_t1"] > thr).astype(int).to_numpy()
y_test = (test["loss_t1"] > thr).astype(int).to_numpy()
X_train = train[feat_cols].to_numpy()
X_test = test[feat_cols].to_numpy()

print(f"=== Label threshold on train (quantile={label_q}) ===")
print("thr =", float(thr))
print("train positives:", int(y_train.sum()), "/", len(y_train), f"({y_train.mean():.4f})")
print("test positives :", int(y_test.sum()), "/", len(y_test), f"({y_test.mean():.4f})")

# --- Logistic regression baseline ---
logit = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=4000, solver="lbfgs", class_weight="balanced"),
)
logit.fit(X_train, y_train)
proba_logit = logit.predict_proba(X_test)[:, 1]

auc_logit = roc_auc_score(y_test, proba_logit) if len(np.unique(y_test)) == 2 else float("nan")
print("\n[Logit] ROC-AUC:", auc_logit)

print("\n[Logit] Alert-budget metrics")
for k in Ks:
    p_at_k = precision_at_k(y_test, proba_logit, k)
    r_at_k = recall_at_k(y_test, proba_logit, k)
    print(f"K={k:>2d}  Precision@K={p_at_k:.4f}  Recall@K={r_at_k:.4f}")

# --- RandomForest baseline ---
rf = RandomForestClassifier(
    n_estimators=500,
    random_state=0,
    class_weight="balanced_subsample",
    min_samples_leaf=5,
)
rf.fit(X_train, y_train)
proba_rf = rf.predict_proba(X_test)[:, 1]

auc_rf = roc_auc_score(y_test, proba_rf) if len(np.unique(y_test)) == 2 else float("nan")
print("\n[RF] ROC-AUC:", auc_rf)

print("\n[RF] Alert-budget metrics")
for k in Ks:
    p_at_k = precision_at_k(y_test, proba_rf, k)
    r_at_k = recall_at_k(y_test, proba_rf, k)
    print(f"K={k:>2d}  Precision@K={p_at_k:.4f}  Recall@K={r_at_k:.4f}")

=== Label threshold on train (quantile=0.9) ===
thr = 0.012771045393253737
train positives: 81 / 804 (0.1007)
test positives : 8 / 201 (0.0398)

[Logit] ROC-AUC: 0.4533678756476684

[Logit] Alert-budget metrics
K= 5  Precision@K=0.0000  Recall@K=0.0000
K=10  Precision@K=0.0000  Recall@K=0.0000
K=20  Precision@K=0.0000  Recall@K=0.0000
K=50  Precision@K=0.0400  Recall@K=0.2500

[RF] ROC-AUC: 0.694300518134715

[RF] Alert-budget metrics
K= 5  Precision@K=0.0000  Recall@K=0.0000
K=10  Precision@K=0.1000  Recall@K=0.1250
K=20  Precision@K=0.0500  Recall@K=0.1250
K=50  Precision@K=0.0800  Recall@K=0.5000


## 4) Expanding walk-forward evaluation

Single split can be sensitive to the specific period.  
We now do **expanding walk-forward**:

- Fold k:
  - Train uses indices `[0, ..., T_k]` (expands over folds)
  - Test uses the next block `(T_k, ..., T_{k+1}]`
- **Leakage-safe rule:** compute the label threshold **within each fold using fold-train only**.

We will output a fold table with:
- fold number
- fold-specific threshold
- test positive count
- ROC-AUC (if both classes exist)
- Precision@K / Recall@K

In [62]:

def walk_forward_expanding_indices(
    n: int,
    initial_train_frac: float = 0.6,
    n_splits: int = 5,
    min_test_size: int = 50,
):
    """
    Return list of (train_idx, test_idx) with expanding train window.
    """
    init_train = int(n * initial_train_frac)
    remaining = n - init_train
    test_block = max(min_test_size, remaining // n_splits)

    splits = []
    for i in range(n_splits):
        train_end = init_train + i * test_block
        test_start = train_end
        test_end = min(test_start + test_block, n)

        if test_end - test_start < min_test_size:
            break

        tr_idx = np.arange(0, train_end)
        te_idx = np.arange(test_start, test_end)
        splits.append((tr_idx, te_idx))

    return splits


def eval_walk_forward_expanding(
    feat_df: pd.DataFrame,
    feat_cols: list[str],
    label_q: float = 0.90,
    initial_train_frac: float = 0.6,
    n_splits: int = 5,
    ks: list[int] = [10, 20, 50],
):
    """
    Expanding walk-forward evaluation.
    For each fold: compute thr on fold-train only, label fold-train/test with that thr.
    """
    X_all = feat_df[feat_cols].to_numpy()
    loss_t1_all = feat_df["loss_t1"].to_numpy()

    splits = walk_forward_expanding_indices(
        n=len(feat_df),
        initial_train_frac=initial_train_frac,
        n_splits=n_splits,
        min_test_size=50,
    )

    rows = []
    for fold, (tr_idx, te_idx) in enumerate(splits, start=1):
        X_tr, X_te = X_all[tr_idx], X_all[te_idx]

        thr_fold = float(np.quantile(loss_t1_all[tr_idx], label_q))
        y_tr = (loss_t1_all[tr_idx] > thr_fold).astype(int)
        y_te = (loss_t1_all[te_idx] > thr_fold).astype(int)

        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=4000, solver="lbfgs", class_weight="balanced"),
        )
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_te)[:, 1]

        auc = roc_auc_score(y_te, proba) if len(np.unique(y_te)) == 2 else float("nan")

        row = {
            "fold": fold,
            "thr_fold": thr_fold,
            "n_test": len(y_te),
            "pos_test": int(y_te.sum()),
            "pos_rate_test": float(np.mean(y_te)),
            "auc": auc,
        }

        for k in ks:
            row[f"prec@{k}"] = precision_at_k(y_te, proba, k)
            row[f"rec@{k}"] = recall_at_k(y_te, proba, k)

        rows.append(row)

    return pd.DataFrame(rows)


wf = eval_walk_forward_expanding(
    feat_df=feat,
    feat_cols=feat_cols,
    label_q=label_q,
    initial_train_frac=0.6,
    n_splits=5,
    ks=[10, 20, 50],
)

wf

,fold,thr_fold,n_test,pos_test,pos_rate_test,auc,prec@10,rec@10,prec@20,rec@20,prec@50,rec@50
0,1,0.012592,80,7,0.0875,0.547945,0.0,0.000,0.00,0.000,0.12,0.857143
1,2,0.012525,80,9,0.1125,0.427230,0.0,0.000,0.00,0.000,0.10,0.555556
2,3,0.012592,80,8,0.1000,0.743056,0.3,0.375,0.15,0.375,0.14,0.875000
3,4,0.012592,80,2,0.0250,0.314103,0.0,0.000,0.00,0.000,0.02,0.500000
4,5,0.012040,80,5,0.0625,0.426667,0.1,0.200,0.05,0.200,0.04,0.400000


## 5) Summary

- Top-K metrics are closer to how alerting works in practice.
- Walk-forward is a more realistic time-series evaluation than a single split.